In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# --------------------------------------------------
# Setup
# --------------------------------------------------

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# Find the input files automatically
# --------------------------------------------------

ticket_files = list(BASE_DIR.rglob("Ticket.csv"))
team_files = list(BASE_DIR.rglob("teams.xlsx"))

if not ticket_files:
    raise FileNotFoundError(
        "tickets.csv was not found. Please put tickets.csv "
        "inside the project folder."
    )

if not team_files:
    raise FileNotFoundError(
        "teams.xlsx was not found. Please put teams.xlsx "
        "inside the project folder."
    )

tickets_path = ticket_files[0]
teams_path = team_files[0]

print("Tickets file:", tickets_path)
print("Teams file:", teams_path)


# --------------------------------------------------
# Load data
# --------------------------------------------------

tickets = pd.read_csv(tickets_path)
teams = pd.read_excel(teams_path)

print("\nFiles loaded successfully.")
print("Tickets shape:", tickets.shape)
print("Teams shape:", teams.shape)


# --------------------------------------------------
# P1 - Numeric data types
# --------------------------------------------------

tickets["resolution_hours"] = pd.to_numeric(
    tickets["resolution_hours"],
    errors="raise"
)

tickets["satisfaction"] = pd.to_numeric(
    tickets["satisfaction"],
    errors="raise"
)

print("\nColumn types:")
print("resolution_hours:", tickets["resolution_hours"].dtype)
print("satisfaction:", tickets["satisfaction"].dtype)


# --------------------------------------------------
# Remove exact duplicate
# --------------------------------------------------

before = len(tickets)

tickets = tickets.drop_duplicates()

after = len(tickets)

print("\nDuplicate rows removed:", before - after)
print("Tickets after cleaning:", after)

assert len(tickets) == 12, (
    f"Expected 12 tickets after cleaning, but got {len(tickets)}"
)


# --------------------------------------------------
# Merge tickets with teams
# --------------------------------------------------

merged = tickets.merge(
    teams,
    on="team_id",
    how="left"
)

print("\nMerged rows:", len(merged))

assert len(merged) == 12, (
    f"Expected 12 rows after merge, but got {len(merged)}"
)

assert merged["department"].isna().sum() == 0, (
    "Some team_id values could not be matched with teams."
)

print("Exactly 12 rows confirmed.")
print("Unmatched team_id values: 0")


# --------------------------------------------------
# P2 - SLA breach flag
# --------------------------------------------------

merged["breach_flag"] = (
    merged["resolution_hours"] > 24
).astype(int)

print("\nBreach flag created.")


# --------------------------------------------------
# Department summary
# --------------------------------------------------

department_summary = (
    merged
    .groupby("department")
    .agg(
        total_tickets=("team_id", "count"),
        breached_tickets=("breach_flag", "sum")
    )
    .reset_index()
)

department_summary["sla_breach_rate_pct"] = (
    department_summary["breached_tickets"]
    / department_summary["total_tickets"]
    * 100
)

print("\nDepartment Summary:")
print(department_summary.to_string(index=False))


# --------------------------------------------------
# Team breach-rate summary
# --------------------------------------------------

team_summary = (
    merged
    .groupby("team_id")
    .agg(
        total_tickets=("team_id", "count"),
        breached_tickets=("breach_flag", "sum")
    )
    .reset_index()
)

team_summary["sla_breach_rate_pct"] = (
    team_summary["breached_tickets"]
    / team_summary["total_tickets"]
    * 100
)

print("\nTeam Summary:")
print(team_summary.to_string(index=False))


# --------------------------------------------------
# Team with highest breach rate
# --------------------------------------------------

highest_team = team_summary.loc[
    team_summary["sla_breach_rate_pct"].idxmax()
]

print("\nTeam with highest SLA breach rate:")
print("Team ID:", highest_team["team_id"])
print(
    "Breached tickets (numerator):",
    int(highest_team["breached_tickets"])
)
print(
    "Total tickets (denominator):",
    int(highest_team["total_tickets"])
)
print(
    "SLA breach rate:",
    round(highest_team["sla_breach_rate_pct"], 2),
    "%"
)


# --------------------------------------------------
# P3 - Find month column
# --------------------------------------------------

possible_month_columns = [
    "month",
    "Month",
    "date",
    "Date",
    "ticket_date",
    "created_at",
    "created_date"
]

month_column = None

for column in possible_month_columns:
    if column in merged.columns:
        month_column = column
        break

if month_column is None:
    raise ValueError(
        "No month/date column was found in tickets.csv. "
        f"Available columns are: {list(merged.columns)}"
    )

print("\nMonth/date column found:", month_column)


# --------------------------------------------------
# Create Jan -> Feb -> Mar order
# --------------------------------------------------

if month_column.lower() == "month":

    merged["month"] = (
        merged[month_column]
        .astype(str)
        .str.strip()
        .str[:3]
    )

else:

    merged[month_column] = pd.to_datetime(
        merged[month_column],
        errors="raise"
    )

    merged["month"] = merged[month_column].dt.strftime("%b")


month_order = ["Jan", "Feb", "Mar"]

merged["month"] = pd.Categorical(
    merged["month"],
    categories=month_order,
    ordered=True
)


# --------------------------------------------------
# Monthly average resolution hours
# --------------------------------------------------

monthly_avg = (
    merged
    .groupby("month", observed=False)["resolution_hours"]
    .mean()
    .reindex(month_order)
)

print("\nMonthly Average Resolution Hours:")
print(monthly_avg)


# --------------------------------------------------
# Create chart
# --------------------------------------------------

plt.figure(figsize=(8, 5))

monthly_avg.plot(
    kind="bar",
    color="steelblue"
)

plt.title("Monthly Average Resolution Hours")
plt.xlabel("Month")
plt.ylabel("Average Resolution Hours")
plt.xticks(rotation=0)

plt.tight_layout()

chart_path = OUTPUT_DIR / "python_chart.png"

plt.savefig(
    chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

merged.to_csv(
    OUTPUT_DIR / "clean_tickets.csv",
    index=False
)

department_summary.to_csv(
    OUTPUT_DIR / "python_summary.csv",
    index=False
)


print("\n--------------------------------")
print("Analysis completed successfully.")
print("--------------------------------")
print("Created:")
print("✓ outputs/python_chart.png")
print("✓ outputs/clean_tickets.csv")
print("✓ outputs/python_summary.csv")


FileNotFoundError: tickets.csv was not found. Please put tickets.csv inside the project folder.